- Automated processing script for lolipop_chart_part2  
- Do not split the main cell

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

%matplotlib inline

In [ ]:
# Main plotting logic
def lolipop_plot(linex,x,dataframe,title,fig_path,ylim = 0,is_save = False):
    plt.close('all')
    fig,ax = plt.subplots(figsize=(30,10))
    if linex is None or len(linex) == 0:
        linex = np.arange(len(dataframe.index))
    linex = np.arange(len(dataframe.index))
    if x is None or len(x) == 0:
        x = linex
    ax.vlines(x=x,ymin=0,ymax=dataframe,color='firebrick',alpha=0.7,linewidth=2)
    ax.scatter(x=x,y=dataframe,s=75,color='firebrick',alpha=0.7)
    
    ax.set_xticklabels(dataframe.index)
    #print(dataframe.index)
    ylim = ylim if ylim != 0 else max(dataframe.max())
    ax.set_ylim(0,ylim)
    ax.set_title(title,fontdict={'size':22})

    plt.show()

    Lolipop_fig = ax.get_figure()
    if is_save:
        Lolipop_fig.savefig(fig_path+'/'+ title + '.pdf',dpi = 100)
        print(f"Figure saved to {fig_path+'/'+ title + '.pdf'}")
    return fig,ax

In [ ]:
# Main batch processing logic
def process_lollipop_charts(projectpath,model,cell_type,test,IDs):
    for TEST1_ID in IDs:
        if isinstance(TEST1_ID, int):
            TEST1_ID = str(TEST1_ID)
        print(f"Processing {cell_type}_{TEST1_ID}...")

        # Define paths
        summary_path = os.path.join(projectpath, "summary")
        signal_path = os.path.join(summary_path, model, cell_type, TEST1_ID,test)
        fig_path = os.path.join(summary_path, model, cell_type, TEST1_ID,"lolipop_chart")

        # Check whether paths exist
        if not os.path.exists(signal_path):
            print(f"Signal path not found for {cell_type}_{TEST1_ID}, skipping...")
            continue
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)

        # Load data
        try:
            df_popchart = pd.read_csv(os.path.join(signal_path, "signal_save", "analysis_total.csv"))
        except FileNotFoundError:
            print(os.path.join(signal_path, "signal_save", "analysis_total.csv"))
            print(f"Analysis file not found for {cell_type}_{TEST1_ID}, skipping...")
            continue

        # Process data
        df_SI = df_popchart[['S_to_I_up', 'S_to_I_down']]
        df_IS = df_popchart[['I_to_S_up', 'I_to_S_down']]

        # Sort the SI chart and reindex the IS chart
        df_SI_sort = df_SI.sort_values(by=['S_to_I_up', 'S_to_I_down'])
        df_IS_sort = df_IS.reindex(df_SI_sort.index)

        # Calculate ylim
        ylim = max(max(df_SI.max()), max(df_IS.max()))
        ylim += 1
        print(f"ylim = {ylim}")
        # Generate S_TO_I charts
        df_S_to_I_up = pd.DataFrame(df_SI_sort['S_to_I_up'])
        df_S_to_I_down = pd.DataFrame(df_SI_sort['S_to_I_down'])

        lolipop_plot(None, None, df_S_to_I_up['S_to_I_up'], f"lolipop_chart_S_to_I_up", fig_path, ylim, True)
        lolipop_plot(None, None, df_S_to_I_down['S_to_I_down'], f"lolipop_chart_S_to_I_down", fig_path, ylim, True)

        # Generate I_TO_S charts
        df_I_to_S_up = pd.DataFrame(df_IS_sort['I_to_S_up'])
        df_I_to_S_down = pd.DataFrame(df_IS_sort['I_to_S_down'])

        lolipop_plot(None, None, df_I_to_S_up['I_to_S_up'], f"lolipop_chart_I_to_S_up", fig_path, ylim, True)
        lolipop_plot(None, None, df_I_to_S_down['I_to_S_down'], f"lolipop_chart_I_to_S_down", fig_path, ylim, True)

        # Generate reordered I_TO_S charts
        df_IS_sort = df_IS.sort_values(by=['I_to_S_up', 'I_to_S_down'])
        df_I_to_S_up_resorted = pd.DataFrame(df_IS_sort['I_to_S_up'])
        df_I_to_S_down_resorted = pd.DataFrame(df_IS_sort['I_to_S_down'])

        lolipop_plot(None, None, df_I_to_S_up_resorted['I_to_S_up'], "lolipop_chart_I_to_S_up_resorted",  fig_path,ylim, True)
        lolipop_plot(None, None, df_I_to_S_down_resorted['I_to_S_down'], "lolipop_chart_I_to_S_down_resorted",  fig_path,ylim, True)

        print(f"Finished processing {cell_type}_{TEST1_ID}")


In [ ]:
#Main program
#Subject placeholders per cell type: VIP (4); CamkII_TST (10); SST (3); PV (7); Hsyn (5)
projectpath = 'path to your projectpath fold'  # Project root containing the signal_data/, behavioral_data/, summary/, and correlation_analysis/ folders used below.
cell_type = 'rescue' 
model = 'test3'
test = 'TST'
type_path = os.path.join(projectpath, 'signal_data', model, cell_type)
IDs = [f for f in os.listdir(type_path) if os.path.isdir(os.path.join(type_path, f))]
print(f"Found {len(IDs)} IDs in {type_path}: {IDs}")
process_lollipop_charts(projectpath,model,cell_type,test, IDs)